# 03 — Load TKG into Neo4j

**Goal:** Load the preprocessed 3W data into Neo4j as a Temporal Knowledge Graph.

## Graph Schema:
- `(Well)-[:HAS_SENSOR]->(Sensor)`
- `(Sensor)-[:MADE_OBSERVATION]->(Observation)`  
- `(Observation)-[:DETECTED_ANOMALY]->(AnomalyEvent)`
- `(Sensor)-[:PRECEDES]->(Sensor)` — physical order downhole→surface
- `(Well)-[:CAUSALLY_COUPLED]->(Well)` — same platform

In [ ]:
import pandas as pd
import json
from pathlib import Path
from neo4j import GraphDatabase
from datetime import datetime, timezone
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

NEO4J_URI      = 'bolt://172.22.43.151:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'your_password'
BATCH_SIZE     = 1000
PROCESSED_DIR  = Path('../../data/processed')

# Test connection
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run('RETURN 1 AS test').single()
    print('✅ Neo4j connected')
except Exception as e:
    print(f'❌ Neo4j connection failed: {e}')

## 1. Create Constraints & Indexes

In [ ]:
def create_constraints(session):
    queries = [
        'CREATE CONSTRAINT well_id IF NOT EXISTS FOR (w:Well) REQUIRE w.id IS UNIQUE',
        'CREATE CONSTRAINT sensor_id IF NOT EXISTS FOR (s:Sensor) REQUIRE s.id IS UNIQUE',
        'CREATE INDEX obs_valid_from IF NOT EXISTS FOR (o:Observation) ON (o.valid_from)',
        'CREATE INDEX obs_event_type IF NOT EXISTS FOR (o:Observation) ON (o.event_type)',
        'CREATE INDEX anomaly_type IF NOT EXISTS FOR (a:AnomalyEvent) ON (a.anomaly_type)',
    ]
    for q in queries:
        try:
            session.run(q)
        except Exception as e:
            print(f'  ⚠️  {e}')
    print('✅ Constraints and indexes created')

with driver.session() as session:
    create_constraints(session)

## 2. Create Wells and Sensors

In [ ]:
def create_wells_and_sensors(session, df: pd.DataFrame, sosa_mapping: dict, sensor_order: list):
    """
    Creates Well and Sensor nodes with SOSA annotation and PRECEDES relations.
    
    PRECEDES relation: physical order PDG→TPT→PCK
    Ref: Vargas et al. (2019) — downhole to surface propagation
    """
    tx_time = datetime.now(timezone.utc).isoformat()
    wells = df['well_id'].unique()

    for well_id in wells:
        session.run('''
            MERGE (w:Well {id: $id})
            SET w.tx_time = $tx_time, w.dataset = '3W'
        ''', id=well_id, tx_time=tx_time)

    # Create sensors with SOSA mapping
    for sensor_id, sosa in sosa_mapping.items():
        session.run('''
            MERGE (s:Sensor {id: $id})
            SET s.observable_property = $prop,
                s.unit = $unit,
                s.position = $position,
                s.tx_time = $tx_time
        ''', id=sensor_id, prop=sosa['observable_property'],
             unit=sosa['unit'], position=sosa['position'], tx_time=tx_time)

    # Create HAS_SENSOR relations
    for well_id in wells:
        for sensor_id in sosa_mapping.keys():
            session.run('''
                MATCH (w:Well {id: $well_id}), (s:Sensor {id: $sensor_id})
                MERGE (w)-[:HAS_SENSOR]->(s)
            ''', well_id=well_id, sensor_id=sensor_id)

    # Create PRECEDES relations (Decision 1)
    for i in range(len(sensor_order) - 1):
        session.run('''
            MATCH (s1:Sensor {id: $src}), (s2:Sensor {id: $dst})
            MERGE (s1)-[:PRECEDES {justification: 'physical_propagation_downhole_to_surface'}]->(s2)
        ''', src=sensor_order[i], dst=sensor_order[i+1])

    print(f'✅ Created {len(wells)} wells, {len(sosa_mapping)} sensors')
    print(f'   PRECEDES chain: {" → ".join(sensor_order)}')

# Load processed data
obs_path = PROCESSED_DIR / 'observations_3w.parquet'
sosa_path = PROCESSED_DIR / 'sosa_mapping.json'
order_path = PROCESSED_DIR / 'sensor_order.json'

if obs_path.exists():
    df = pd.read_parquet(obs_path)
    with open(sosa_path) as f:
        sosa_mapping = json.load(f)
    with open(order_path) as f:
        sensor_order = json.load(f)['precedes_order']

    with driver.session() as session:
        create_wells_and_sensors(session, df, sosa_mapping, sensor_order)
else:
    print('⚠️  Run 02_preprocessing.ipynb first')

## 3. Load Observations (Batched)

In [ ]:
def load_observations(session, df: pd.DataFrame, sensors: list):
    """Loads observations with bitemporal annotation in batches."""
    tx_time = datetime.now(timezone.utc).isoformat()
    total = 0

    for sensor_id in sensors:
        if sensor_id not in df.columns:
            continue
        sensor_df = df[['well_id','valid_from','tx_time','quality_flag',
                         'event_type', sensor_id]].copy()
        sensor_df = sensor_df.rename(columns={sensor_id: 'value'})
        sensor_df['sensor_id'] = sensor_id
        sensor_df['is_anomaly'] = df['class'] > 0

        records = sensor_df.dropna(subset=['value']).to_dict('records')

        for i in range(0, len(records), BATCH_SIZE):
            batch = records[i:i+BATCH_SIZE]
            session.run('''
                UNWIND $rows AS row
                MATCH (s:Sensor {id: row.sensor_id})
                CREATE (o:Observation {
                    sensor_id:     row.sensor_id,
                    well_id:       row.well_id,
                    value:         row.value,
                    valid_from:    row.valid_from,
                    valid_to:      null,
                    tx_time:       row.tx_time,
                    quality_flag:  row.quality_flag,
                    event_type:    row.event_type,
                    is_anomaly:    row.is_anomaly
                })
                CREATE (s)-[:MADE_OBSERVATION {valid_from: row.valid_from}]->(o)
                WITH o, row
                WHERE row.is_anomaly = true
                CREATE (a:AnomalyEvent {
                    sensor_id:    row.sensor_id,
                    anomaly_type: row.event_type,
                    valid_from:   row.valid_from,
                    tx_time:      row.tx_time
                })
                CREATE (o)-[:DETECTED_ANOMALY]->(a)
            ''', rows=batch)
            total += len(batch)

        print(f'  ✅ {sensor_id}: {len(records):,} observations loaded')

    print(f'\n✅ Total: {total:,} observations loaded')

if obs_path.exists():
    SENSORS = list(sosa_mapping.keys())
    with driver.session() as session:
        load_observations(session, df, SENSORS)
else:
    print('⚠️  Run 02_preprocessing.ipynb first')

## 4. Create CAUSALLY_COUPLED Relations

In [ ]:
def create_causally_coupled(session, df: pd.DataFrame):
    """
    Creates CAUSALLY_COUPLED relations between wells.

    Design Decision 2: wells sharing the same platform are coupled.
    In 3W dataset: wells with same prefix (WELL-0001x) are on same platform.
    Ref: Diamantini et al. [25] — process-aware IIoT knowledge graph
    """
    wells = df['well_id'].unique().tolist()
    # Group wells by platform prefix (first 8 chars of well name)
    platforms = {}
    for w in wells:
        platform = w[:8] if len(w) >= 8 else w
        platforms.setdefault(platform, []).append(w)

    coupled = 0
    for platform, platform_wells in platforms.items():
        for i in range(len(platform_wells)):
            for j in range(i+1, len(platform_wells)):
                session.run('''
                    MATCH (w1:Well {id: $w1}), (w2:Well {id: $w2})
                    MERGE (w1)-[:CAUSALLY_COUPLED {
                        justification: 'same_platform',
                        ref: 'Diamantini_et_al_2023'
                    }]->(w2)
                ''', w1=platform_wells[i], w2=platform_wells[j])
                coupled += 1

    print(f'✅ Created {coupled} CAUSALLY_COUPLED relations')

if obs_path.exists():
    with driver.session() as session:
        create_causally_coupled(session, df)
else:
    print('⚠️  Run 02_preprocessing.ipynb first')

## 5. Verify Graph

In [ ]:
with driver.session() as session:
    for label in ['Well','Sensor','Observation','AnomalyEvent']:
        count = session.run(f'MATCH (n:{label}) RETURN count(n) AS c').single()['c']
        print(f'  {label:<15}: {count:,}')

    for rel in ['HAS_SENSOR','MADE_OBSERVATION','DETECTED_ANOMALY','PRECEDES','CAUSALLY_COUPLED']:
        count = session.run(f'MATCH ()-[r:{rel}]->() RETURN count(r) AS c').single()['c']
        print(f'  :{rel:<25}: {count:,}')

driver.close()